How to record and save video of Gym environment

https://stackoverflow.com/questions/77042526/how-to-record-and-save-video-of-gym-environment

manual control replay

In [1]:
import gymnasium as gym
import numpy as np
import csv
import time
from gym.wrappers import RecordVideo
import os

In [1]:
def replay_from_csv(csv_file, video_path='video'):
    # 初始化MountainCarContinuous环境并包装RecordVideo
    env = gym.make('MountainCarContinuous-v0',render_mode="rgb_array")
    
    env = RecordVideo(env, video_path)

    env.reset()

    # 读取CSV文件
    with open(csv_file, newline='') as file:
        reader = csv.DictReader(file)
        
        for row in reader:
            # 从CSV行中获取动作和状态
            action = float(row['action'])
            position = float(row['position'])
            velocity = float(row['velocity'])
            
            # 设置环境状态 (注意：此步骤仅为演示用途，通常不建议直接设置状态)
            env.env.state = (position, velocity)
            
            # 执行动作
            env.step([action])  # 动作是一个浮点数列表
            
            # 模拟时间流逝 (根据时间戳调整播放速度)
            time.sleep(0.1)

    env.close()

if __name__ == '__main__':
    replay_from_csv('episode_1.csv')

c:\Users\syf26\.conda\envs\dif_aug_cuda\lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.is_vector_env to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.is_vector_env` for environment variables or `env.get_wrapper_attr('is_vector_env')` that will search the reminding wrappers.
  logger.warn(


Moviepy - Building video d:\s10\NAIST\Codes\GymScripts\video\rl-video-episode-0.mp4.
Moviepy - Writing video d:\s10\NAIST\Codes\GymScripts\video\rl-video-episode-0.mp4



Moviepy - Done !
Moviepy - video ready d:\s10\NAIST\Codes\GymScripts\video\rl-video-episode-0.mp4


## Create videos for fixed expolicy expert demo data

In [4]:
# 读取CSV并按episode拆分
def split_csv_by_episode(filepath):
    episodes_dict = {}  # 用于存储每个episode的数据
    
    # 读取CSV文件
    with open(filepath, 'r') as csvfile:
        reader = csv.DictReader(csvfile)
        
        # 遍历每一行数据
        for row in reader:
            episode = int(row['episode'])  # 当前行的 episode
            step = int(row['step'])        # 当前行的 step
            position = float(row['position'])  # 当前行的 position
            velocity = float(row['velocity'])  # 当前行的 velocity
            action = float(row['action'])      # 当前行的 action
            
            # 如果该 episode 不在字典中，初始化一个空数组
            if episode not in episodes_dict:
                episodes_dict[episode] = []
            
            # 将当前行的数据按照顺序加入对应 episode 的数组
            episodes_dict[episode].append([step, position, velocity, action])
    
    return episodes_dict

# 示例使用方法
if __name__ == "__main__":
    filepath = './data/fixed_policy_data_2.csv'
    episodes_data = split_csv_by_episode(filepath)  # 拆分后的数据
    
    # 输出每个 episode 的数据
    for episode, data in episodes_data.items():
        print(f"Episode {episode}:")
        for step_data in data:
            print(step_data)

Episode 1:
[1, -0.44566956, -0.002094429, -1.0]
[2, -0.44984314, -0.0041735885, -1.0]
[3, -0.4560654, -0.006222253, -1.0]
[4, -0.46429068, -0.008225296, -1.0]
[5, -0.47445846, -0.010167764, -1.0]
[6, -0.48649344, -0.012034982, -1.0]
[7, -0.5003061, -0.013812698, -1.0]
[8, -0.5157934, -0.015487251, -1.0]
[9, -0.5328392, -0.017045787, -1.0]
[10, -0.55131567, -0.018476492, -1.0]
[11, -0.5710845, -0.019768855, -1.0]
[12, -0.5919984, -0.020913916, -1.0]
[13, -0.6139029, -0.021904511, -1.0]
[14, -0.6366384, -0.022735484, -1.0]
[15, -0.6600422, -0.023403844, -1.0]
[16, -0.6839511, -0.023908855, -1.0]
[17, -0.70820314, -0.024252065, -1.0]
[18, -0.7326404, -0.024437228, -1.0]
[19, -0.7571106, -0.02447018, -1.0]
[20, -0.7814692, -0.024358612, -1.0]
[21, -0.80558103, -0.024111804, -1.0]
[22, -0.8293213, -0.023740306, -1.0]
[23, -0.8525769, -0.0232556, -1.0]
[24, -0.87524664, -0.02266975, -1.0]
[25, -0.8972417, -0.021995068, -1.0]
[26, -0.9184855, -0.021243805, -1.0]
[27, -0.9389134, -0.020427877,

### Split CSV and all replay in gymnasium 

In [6]:
# 定义重放函数
def replay_episode(env, episode_data, video_path):
    # 初始化RecordVideo来记录每个episode
    env = RecordVideo(env, video_path)

    # 重置环境
    env.reset()

    for step_data in episode_data:
        # 从数据中获取action, position, velocity
        action = float(step_data['action'])
        position = float(step_data['position'])
        velocity = float(step_data['velocity'])
        
        # 设置环境的状态 (position, velocity)
        env.env.state = (position, velocity)
        
        # 执行动作
        env.step([action])  # 动作是一个浮点数列表
        
        # 模拟时间流逝
        time.sleep(0.1)
    
    env.close()

# 读取CSV并按 episode 组织数据
def load_csv_data_by_episode(csv_file):
    episodes_dict = {}
    
    with open(csv_file, newline='') as file:
        reader = csv.DictReader(file)
        
        for row in reader:
            episode = int(row['episode'])
            
            if episode not in episodes_dict:
                episodes_dict[episode] = []
            
            # 将每一行的数据保存到相应的episode中
            episodes_dict[episode].append(row)
    
    return episodes_dict

# 主函数，重放所有 episodes 并生成视频
def replay_from_csv(csv_file, video_dir):
    # 创建视频保存目录
    if not os.path.exists(video_dir):
        os.makedirs(video_dir)
    
    # 加载 CSV 数据
    episodes_dict = load_csv_data_by_episode(csv_file)

    # 初始化 MountainCarContinuous 环境
    env = gym.make('MountainCarContinuous-v0', render_mode="rgb_array")
    
    for episode, episode_data in episodes_dict.items():
        print(f"Replaying episode {episode}")
        video_path = os.path.join(video_dir, f"episode_{episode}")
        
        # 重放每个 episode
        replay_episode(env, episode_data, video_path)

    env.close()

if __name__ == '__main__':
    #replay_from_csv('your_csv_file.csv')  # 替换为你的CSV文件路径

    # 运行重现代码
    csv_filepath = './data/manual_policy_100_LR_RLR.csv'  # 替换为你的 CSV 文件路径
    output_dir = 'videos/'  # 视频保存目录
    replay_from_csv(csv_filepath, output_dir)

Replaying episode 1
Moviepy - Building video d:\s10\NAIST\Codes\GymScripts\videos\episode_1\rl-video-episode-0.mp4.
Moviepy - Writing video d:\s10\NAIST\Codes\GymScripts\videos\episode_1\rl-video-episode-0.mp4



Moviepy - Done !
Moviepy - video ready d:\s10\NAIST\Codes\GymScripts\videos\episode_1\rl-video-episode-0.mp4
Replaying episode 2
Moviepy - Building video d:\s10\NAIST\Codes\GymScripts\videos\episode_2\rl-video-episode-0.mp4.
Moviepy - Writing video d:\s10\NAIST\Codes\GymScripts\videos\episode_2\rl-video-episode-0.mp4



Moviepy - Done !
Moviepy - video ready d:\s10\NAIST\Codes\GymScripts\videos\episode_2\rl-video-episode-0.mp4


Replaying episode 3
Moviepy - Building video d:\s10\NAIST\Codes\GymScripts\videos\episode_3\rl-video-episode-0.mp4.
Moviepy - Writing video d:\s10\NAIST\Codes\GymScripts\videos\episode_3\rl-video-episode-0.mp4



Moviepy - Done !
Moviepy - video ready d:\s10\NAIST\Codes\GymScripts\videos\episode_3\rl-video-episode-0.mp4
Replaying episode 4


...

Moviepy - Done !
Moviepy - video ready d:\s10\NAIST\Codes\GymScripts\videos\episode_98\rl-video-episode-0.mp4


Replaying episode 99
Moviepy - Building video d:\s10\NAIST\Codes\GymScripts\videos\episode_99\rl-video-episode-0.mp4.
Moviepy - Writing video d:\s10\NAIST\Codes\GymScripts\videos\episode_99\rl-video-episode-0.mp4



Moviepy - Done !
Moviepy - video ready d:\s10\NAIST\Codes\GymScripts\videos\episode_99\rl-video-episode-0.mp4


Replaying episode 100
Moviepy - Building video d:\s10\NAIST\Codes\GymScripts\videos\episode_100\rl-video-episode-0.mp4.
Moviepy - Writing video d:\s10\NAIST\Codes\GymScripts\videos\episode_100\rl-video-episode-0.mp4



Moviepy - Done !
Moviepy - video ready d:\s10\NAIST\Codes\GymScripts\videos\episode_100\rl-video-episode-0.mp4
